In [ ]:
!pip install groq


In [ ]:
import csv
import json
import os
import re
import time
from groq import Groq

# ==============================================================================
# 1. CONFIGURAÇÃO OTIMIZADA COM OS MODELOS DA SUA TELA
# ==============================================================================
GROQ_API_KEY = ""  # Começa com gsk_...
ARQUIVO_ENTRADA = "issues.csv"
ARQUIVO_SAIDA_CSV = "resultados_requisitos.csv"
ARQUIVO_SAIDA_MD = "relatorio_completo_requisitos.md"

# Deixe None para rodar todas as 150 issues economizando cota
LIMITE_TESTE = None

client = Groq(api_key=GROQ_API_KEY)

# Ajustado rigorosamente para os modelos ativos na sua imagem
MODELO_GERADOR = "openai/gpt-oss-120b"  # Usa o modelo maior (cota nova e zerada)
MODELO_JUIZ = "qwen/qwen3.8-27b"        # Mantém o Qwen como Juiz independente

def limpar_raciocinio(texto: str) -> str:
    """Remove blocos internos <think>...</think> dos modelos de raciocínio."""
    return re.sub(r"<think>.*?</think>", "", texto, flags=re.DOTALL).strip()

def chamar_groq(modelo: str, prompt_sistema: str, mensagem_usuario: str, json_mode: bool = False) -> str:
    """Chama a API da Groq com tratamento de erros e pausa de segurança."""
    for _ in range(3):
        try:
            parametros = {
                "model": modelo,
                "messages": [
                    {"role": "system", "content": prompt_sistema},
                    {"role": "user", "content": mensagem_usuario},
                ],
                "temperature": 0.4,
            }
            if json_mode:
                parametros["response_format"] = {"type": "json_object"}

            resposta = client.chat.completions.create(**parametros)
            conteudo = resposta.choices[0].message.content or ""
            time.sleep(2.5)  # Pausa essencial para não estourar o limite por minuto (RPM)
            return limpar_raciocinio(conteudo)
        except Exception as e:
            print(f"    ⚠️ Aguardando instabilidade da API: {str(e)[:100]}")
            time.sleep(15)
    return ""

def extrair_json(texto: str) -> dict:
    texto_limpo = limpar_raciocinio(texto)
    match = re.search(r"\{.*\}", texto_limpo, re.DOTALL)
    if match:
        return json.loads(match.group(0))
    raise ValueError("JSON inválido")

# ==============================================================================
# 2. PROMPTS COMPACTADOS (MÁXIMA ECONOMIA DE TOKENS)
# ==============================================================================

PROMPT_GERADOR_COMPACTO = """
Você é um engenheiro de requisitos. Escreva um único requisito de software em português a partir do título da issue enviada.
Diretrizes ISO/IEC/IEEE 29148 a cumprir obrigatoriamente:
1. Comece com "O sistema deve...".
2. Sem ambiguidade: Linguagem técnica exata e direta.
3. Singularidade: Apenas UMA capacidade em uma única frase curta (máximo 30 palavras).
4. Verificabilidade: Deve ser possível testar de forma binária (passa/falha).

Responda APENAS com a frase do requisito consolidado, sem explicações ou introduções.
"""

PROMPT_JUIZ_COMPACTO = """
Avalie o requisito nos critérios ISO 29148: unambiguity, singularity, verifiability. Atribua notas de 1 a 5.
Responda EXCLUSIVAMENTE em formato JSON:
{
  "unambiguity": <1-5>,
  "singularity": <1-5>,
  "verifiability": <1-5>,
  "explanation": "<motivo curto se alguma nota for < 4, ou 'Aprovado'>"
}
"""

# ==============================================================================
# 3. FLUXO DE PROCESSAMENTO EM LOTE OTIMIZADO
# ==============================================================================

def processar_issue(repo: str, issue_num: str, issue_title: str, max_tentativas: int = 3) -> dict:
    feedback_juiz = ""
    melhor_resultado = {
        "repository": repo, "issue_number": issue_num, "issue_title": issue_title,
        "status": "REJECT", "iteracoes": max_tentativas, "requisito_final": "",
        "unambiguity": 0, "singularity": 0, "verifiability": 0, "parecer_juiz": ""
    }

    for tentativa in range(1, max_tentativas + 1):
        contexto = f"Issue: {issue_title}"
        if feedback_juiz:
            contexto += f"\nCorrija a falha apontada pelo Juiz: {feedback_juiz}. Remova detalhes extras para manter focado e singular."

        # O GPT OSS 120B gera o requisito de forma direta e atômica
        requisito_final = chamar_groq(MODELO_GERADOR, PROMPT_GERADOR_COMPACTO, contexto)

        # O Qwen 27B atua como o Juiz independente
        dados_juiz = chamar_groq(MODELO_JUIZ, PROMPT_JUIZ_COMPACTO, f"Req: {requisito_final}", json_mode=True)

        try:
            avaliacao = extrair_json(dados_juiz)
            u, s, v = avaliacao["unambiguity"], avaliacao["singularity"], avaliacao["verifiability"]
            parecer = avaliacao["explanation"]

            soma_atual = u + s + v
            soma_melhor = melhor_resultado["unambiguity"] + melhor_resultado["singularity"] + melhor_resultado["verifiability"]

            if soma_atual >= soma_melhor:
                melhor_resultado.update({
                    "iteracoes": tentativa, "requisito_final": requisito_final,
                    "unambiguity": u, "singularity": s, "verifiability": v, "parecer_juiz": parecer
                })

            if u >= 4 and s >= 4 and v >= 4:
                melhor_resultado["status"] = "APPROVE"
                return melhor_resultado
            else:
                feedback_juiz = parecer
        except:
            feedback_juiz = "Torne o requisito mais curto, simples e fácil de testar."

    return melhor_resultado

def carregar_ja_processados(caminho_csv: str) -> set:
    processados = set()
    if os.path.exists(caminho_csv):
        with open(caminho_csv, mode="r", encoding="utf-8") as f:
            for linha in csv.DictReader(f):
                processados.add((linha["repository"].strip(), linha["issue_number"].strip()))
    return processados

def executar_lote():
    if not os.path.exists(ARQUIVO_ENTRADA):
        print(f"❌ Erro: Arquivo '{ARQUIVO_ENTRADA}' não encontrado na aba lateral.")
        return

    with open(ARQUIVO_ENTRADA, mode="r", encoding="utf-8") as f:
        issues = list(csv.DictReader(f))

    if LIMITE_TESTE:
        issues = issues[:LIMITE_TESTE]

    ja_processados = carregar_ja_processados(ARQUIVO_SAIDA_CSV)
    arquivo_existe = os.path.exists(ARQUIVO_SAIDA_CSV) and len(ja_processados) > 0

    modo_abertura = "a" if arquivo_existe else "w"
    campos_saida = ["repository", "issue_number", "issue_title", "status", "iteracoes", "requisito_final", "unambiguity", "singularity", "verifiability", "parecer_juiz"]

    with open(ARQUIVO_SAIDA_CSV, mode=modo_abertura, newline="", encoding="utf-8") as f_csv, \
         open(ARQUIVO_SAIDA_MD, mode=modo_abertura, encoding="utf-8") as f_md:

        escritor = csv.DictWriter(f_csv, fieldnames=campos_saida)
        if not arquivo_existe:
            escritor.writeheader()
            f_md.write("# Relatório de Requisitos Gerados (Modelos OpenAI e Qwen)\n\n")

        total = len(issues)
        for idx, linha in enumerate(issues, start=1):
            repo, num, titulo = linha["Repository"].strip(), linha["Issue Number"].strip(), linha["Issue Title"].strip()
            if (repo, num) in ja_processados: continue

            print(f"[{idx}/{total}] 📌 {repo} #{num}...", end="")
            resultado = processar_issue(repo, num, titulo)
            escritor.writerow(resultado)
            f_csv.flush()

            print(f" Status: {resultado['status']} (Notas: {resultado['unambiguity']},{resultado['singularity']},{resultado['verifiability']})")

            if resultado["status"] == "APPROVE":
                f_md.write(f"### ✅ [APROVADO] {repo} #{num}\n> {resultado['requisito_final']}\n\n")
            else:
                f_md.write(f"### ⚠️ [REVISÃO] {repo} #{num}\n> {resultado['requisito_final']}\n- *Parecer:* {resultado['parecer_juiz']}\n\n")
            f_md.flush()

    print("\n🏁 CONCLUÍDO COM OS MODELOS DO SEU PAINEL!")

executar_lote()


[1/150] 📌 pytorch/pytorch #77764... Status: APPROVE (Notas: 5,5,5)
[2/150] 📌 pytorch/pytorch #47688... Status: APPROVE (Notas: 5,5,5)
[3/150] 📌 pytorch/pytorch #17835... Status: APPROVE (Notas: 4,5,4)
[4/150] 📌 pytorch/pytorch #2001... Status: APPROVE (Notas: 5,5,5)
[5/150] 📌 pytorch/pytorch #26340... Status: APPROVE (Notas: 4,5,4)
[6/150] 📌 pytorch/pytorch #16897... Status: APPROVE (Notas: 5,5,5)
[7/150] 📌 pytorch/pytorch #38419... Status: APPROVE (Notas: 5,5,5)
[8/150] 📌 pytorch/pytorch #23430... Status: APPROVE (Notas: 5,5,5)
[9/150] 📌 pytorch/pytorch #25032... Status: APPROVE (Notas: 4,5,5)
[10/150] 📌 pytorch/pytorch #20822... Status: APPROVE (Notas: 5,5,5)
[11/150] 📌 pytorch/pytorch #26889... Status: APPROVE (Notas: 4,4,5)
[12/150] 📌 pytorch/pytorch #49440... Status: REJECT (Notas: 4,3,5)
[13/150] 📌 pytorch/pytorch #24826... Status: APPROVE (Notas: 4,5,4)
[14/150] 📌 pytorch/pytorch #64932... Status: APPROVE (Notas: 4,5,4)
[15/150] 📌 pytorch/pytorch #27610... Status: APPROVE (Notas

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# 1. Carrega os dados gerados pelo seu pipeline
df = pd.read_csv("resultados_requisitos.csv")
atributos = ['unambiguity', 'singularity', 'verifiability']

print("=" * 65)
print("📊 ESTATÍSTICA DESCRITIVA AVANÇADA (IDÊNTICA À TABELA 2 DO ARTIGO)")
print("=" * 65)

# Calcula as métricas clássicas para cada um dos três atributos
resumo = df[atributos].describe().T
resumo['median'] = df[atributos].median()
resumo = resumo[['mean', 'median', 'std', 'min', 'max']]
resumo.columns = ['Média', 'Mediana', 'Desvio Padrão (SD)', 'Mínimo', 'Máximo']

# Formata a exibição para 2 casas decimais
print(resumo.round(2).to_string())
print("\n" + "-" * 65)

print("📈 FREQUÊNCIA ABSOLUTA DAS NOTAS (DISTRIBUIÇÃO DE 1 A 5)")
print("-" * 65)
for col in atributos:
    print(f"\nDistr. de Notas para {col.upper()}:")
    contagem = df[col].value_counts().sort_index()
    for nota in range(1, 6):
        qtd = contagem.get(nota, 0)
        pct = (qtd / len(df)) * 100
        print(f"  Nota {nota}: {qtd} ocorrências ({pct:.1f}%)")

print("\n" + "=" * 65)
print("🔗 MATRIZ DE CORREÇÃO DE SPEARMAN (IDÊNTICA À TABELA 6 DO ARTIGO)")
print("=" * 65)

# Calcula os coeficientes de correlação de Spearman e os p-valores
coef_uv, p_uv = spearmanr(df['unambiguity'], df['verifiability'])
coef_us, p_us = spearmanr(df['unambiguity'], df['singularity'])
coef_sv, p_sv = spearmanr(df['singularity'], df['verifiability'])

print(f"Unambiguity vs Verifiability:   Coef= {coef_uv:.2f} | p-valor= {p_uv:.4f}")
print(f"Unambiguity vs Singularity:     Coef= {coef_us:.2f} | p-valor= {p_us:.4f}")
print(f"Singularity vs Verifiability:   Coef= {coef_sv:.2f} | p-valor= {p_sv:.4f}")

print("\n💡 Interpretação Científica Rápida:")
print("- p-valor < 0.05 significa que a relação é estatisticamente significativa.")
print("- No artigo, Unambiguity e Verifiability possuem forte correlação positiva (0.69).")


📊 ESTATÍSTICA DESCRITIVA AVANÇADA (IDÊNTICA À TABELA 2 DO ARTIGO)
               Média  Mediana  Desvio Padrão (SD)  Mínimo  Máximo
unambiguity     4.41      4.0                0.64     2.0     5.0
singularity     4.64      5.0                0.56     3.0     5.0
verifiability   4.47      5.0                0.64     2.0     5.0

-----------------------------------------------------------------
📈 FREQUÊNCIA ABSOLUTA DAS NOTAS (DISTRIBUIÇÃO DE 1 A 5)
-----------------------------------------------------------------

Distr. de Notas para UNAMBIGUITY:
  Nota 1: 0 ocorrências (0.0%)
  Nota 2: 2 ocorrências (1.3%)
  Nota 3: 6 ocorrências (4.0%)
  Nota 4: 71 ocorrências (47.3%)
  Nota 5: 71 ocorrências (47.3%)

Distr. de Notas para SINGULARITY:
  Nota 1: 0 ocorrências (0.0%)
  Nota 2: 0 ocorrências (0.0%)
  Nota 3: 6 ocorrências (4.0%)
  Nota 4: 42 ocorrências (28.0%)
  Nota 5: 102 ocorrências (68.0%)

Distr. de Notas para VERIFIABILITY:
  Nota 1: 0 ocorrências (0.0%)
  Nota 2: 2 ocorrências 